<a href="https://colab.research.google.com/github/Kate6097/train/blob/important-functions/Direction_loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Direction_loss(torch.nn.Module):
    def __init__(self, stylegan_size=1024):
        super().__init__()
        self.model, _ = clip.load("ViT-B/32", device=device)
        self.resize = Resize((224, 224))
        self.normalize = Normalize(
            mean=[0.48145466, 0.4578275, 0.40821073],
            std=[0.26862954, 0.26130258, 0.27577711]
        )

    def forward(self, t_source, t_target, img_trainable, img_frozen, beta=1):

        # Подготовка для CLIP
        clip_frozen = self.normalize(self.resize(img_frozen))
        clip_trainable = self.normalize(self.resize(img_trainable))

        # Текстовая разница
        T = beta*(self.model.encode_text(t_target) - self.model.encode_text(t_source))

        # Визуальная разница
        I = self.model.encode_image(clip_trainable) - self.model.encode_image(clip_frozen)

        # Косинусное расстояние (нормализация как в clip происходит под капотом)
        cos_sim = torch.cosine_similarity(I, T, dim=-1)
        return 1 - cos_sim.mean()

